In [0]:
from pyspark.sql import functions as F

fct   = spark.read.table("northstar.gold.fct_orders")
items = spark.read.table("northstar.silver.order_items")
cust  = spark.read.table("northstar.gold.dim_customer")

# Order-level features from line items
item_feat = (items.groupBy("order_id").agg(
    F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight"),
    F.count("*").alias("num_items")))

features = (fct
    .filter(F.col("is_delivered") == 1)                  # label only valid for delivered orders
    .filter(F.col("estimated_days").isNotNull())
    .join(item_feat, "order_id", "left")
    .join(cust.select("customer_id","customer_state"), "customer_id", "left")
    .withColumn("order_month", F.month("order_purchase_timestamp"))
    .withColumn("order_dow",   F.dayofweek("order_purchase_timestamp"))
    .select("order_id",
            "estimated_days", "total_price", "total_freight", "num_items",   # ← purchase-time features (no leakage)
            "customer_state", "order_month", "order_dow",
            "is_late")                                                       # ← the label
    .na.drop())

features.write.mode("overwrite").saveAsTable("northstar.gold.ml_features_late_delivery")
print(f"✅ feature table: {features.count()} rows")

In [0]:
import mlflow, mlflow.sklearn, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

pdf = spark.read.table("northstar.gold.ml_features_late_delivery").toPandas()
pdf = pd.get_dummies(pdf, columns=["customer_state"], drop_first=True)

X = pdf.drop(columns=["order_id", "is_late"]).astype("float64")     # ← cast to float (fixes Warning 1)
y = pdf["is_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)               # ← stratify keeps class ratio

mlflow.sklearn.autolog()
with mlflow.start_run(run_name="late_delivery_rf_balanced"):
    model = RandomForestClassifier(n_estimators=200, max_depth=10,
                                   class_weight="balanced", random_state=42)  # ← fixes imbalance (Warning 2)
    model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    mlflow.log_metric("test_auc", auc)
    print(f"✅ Test AUC: {auc:.3f}")
    print(classification_report(y_test, model.predict(X_test)))     # precision/recall per class